In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import math, glob
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM
from transformers import TrainingArguments
from peft import LoraConfig, TaskType

from bait.utils import common_utils, json_utils, model_utils, tokenizer_utils, container_utils
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE
from bait.core import bait_utils
from bait.core.sft_trainer import SftTrainer

In [ ]:
seed = GlobalCommonConfig.SEED
common_utils.set_seed(seed)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_sft_datas'
out_dir = f'{data_dir}/sft'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def get_peft_config(lora_r,
                    lora_alpha,
                    lora_dropout,
                    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                    bias='none',
                    task_type=TaskType.CAUSAL_LM):
    
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias=bias,
        task_type=task_type
    )

    return peft_config

In [ ]:
def get_training_args(out_dir,
                      num_epochs,
                      batch_size,
                      accumulation_steps,
                      learning_rate,
                      weight_decay,
                      warmup_ratio,
                      max_grad_norm,
                      save_strategy='steps',
                      save_steps=10,
                      eval_strategy='steps',
                      eval_steps=10,
                      logging_steps=10,
                      lr_scheduler_type='cosine',
                      load_best_model_at_end=True,
                      metric_for_best_model='eval_loss',
                      save_total_limit=5):
    
    training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        max_grad_norm=max_grad_norm,

        save_strategy=save_strategy,
        save_steps=save_steps,
        eval_strategy=eval_strategy,
        eval_steps=eval_steps,
        logging_steps=logging_steps,

        lr_scheduler_type=lr_scheduler_type,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        save_total_limit=save_total_limit,

        bf16=True,
        do_train=True,
        label_names=['labels'],
        report_to='none'
    )

    return training_args

In [ ]:
model_names = ['Llama-3.2-3B', 'Qwen2.5-3B', 'Llama-3.1-8B', 'Qwen2.5-7B']
zero_shot_types = ['fact', 'counter', 'other']

lora_r, lora_alpha, lora_dropout = 64, 128, 0.05
num_epochs, batch_size, accumulation_steps = 2, 1, 128
learning_rate, weight_decay, warmup_ratio, max_grad_norm = 5e-5, 0.01, 0.05, 1.0
save_and_eval_per_epoch, logging_per_epoch = 5, 5

for model_name in model_names:
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)

    for zero_shot_type in zero_shot_types:
        train_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot_type}_sft_train.json'
        eval_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot_type}_sft_eval.json'

        train_datas = json_utils.load_json(train_file_path)
        eval_datas = json_utils.load_json(eval_file_path)

        real_batch_size = batch_size * accumulation_steps
        steps_per_epoch = math.ceil(len(train_datas) / real_batch_size)
        save_and_eval_steps = max(1, steps_per_epoch // save_and_eval_per_epoch)
        logging_steps = max(1, steps_per_epoch // logging_per_epoch)

        sft_trainer = SftTrainer(model_name_or_path, max_seq_length, dtype, device)

        peft_config = get_peft_config(lora_r, lora_alpha, lora_dropout)
        sft_trainer.init_model(peft_config)

        save_dir = f'{out_dir}/{model_name}/zero_shot_{zero_shot_type}'

        training_args = get_training_args(
            save_dir, num_epochs, batch_size, accumulation_steps,
            learning_rate, weight_decay, warmup_ratio, max_grad_norm,
            save_steps=save_and_eval_steps, eval_steps=save_and_eval_steps, logging_steps=logging_steps
        )

        sft_trainer.set_and_get_sft_dataset(train_datas, eval_datas)
        sft_trainer.train(training_args, early_stopping_patience=3)
        sft_trainer.clear()

In [ ]:
def evaluate(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas: list, batch_size):
    data_size = len(datas)
    cnts, sizes = {}, {}

    for i, datas_batch in enumerate(container_utils.chunks(datas, batch_size)):
        prompts_batch = [data['source']['content'] for data in datas_batch]

        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        for data, generated_text in zip(datas_batch, generated_texts):
            file_format = data['file_format']
            ext_n_fact = data['ext_n_fact']
            ext_n_counter = data['ext_n_counter']
            target = data['target']

            key = f'{file_format}\t{ext_n_fact}\t{ext_n_counter}'
            key_all = f'ALL\t{ext_n_fact}\t{ext_n_counter}'

            # key 저장 용도
            container_utils.add_str_int(cnts, key, 0)
            container_utils.add_str_int(cnts, key_all, 0)

            if model_utils.is_correct(generated_text, target)[1]:
                container_utils.add_str_int(cnts, key, 1)
                container_utils.add_str_int(cnts, key_all, 1)

            # size 저장 용도
            container_utils.add_str_int(sizes, key, 1)
            container_utils.add_str_int(sizes, key_all, 1)

        if (i+1) % 5000 == 0:
            print(f'evaluate() {(i+1)*batch_size} complet.')
    print(f'evaluate() {data_size} complet.\n')





    file_formats = FILE_FORMATS + ['ALL']
    for file_format in file_formats:
        for i in range(CONTEXT_SIZE):
            key = f'{file_format}\t{CONTEXT_SIZE-1-i}\t{i}'

            if key in cnts.keys():
                cnt = cnts[key]
                size = sizes[key]

                print(f'{key}\t{cnt}\t{cnt}/{size}\t{cnt/size}')
        print()

In [ ]:
def evaluate_all(checkpoint_dir, eval_datas, batch_size):
    checkpoint_paths = glob.glob(f'{checkpoint_dir}/checkpoint-*')
    checkpoint_paths.sort(key=lambda x: int(x.split('-')[-1]))

    for checkpoint_path in checkpoint_paths:
        model = model_utils.get_model(checkpoint_path, dtype, device=device, is_eval=True)
        tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

        evaluate(model, tokenizer, eval_datas, batch_size)

        del model
        del tokenizer
        common_utils.clear_gpu_memory()

In [ ]:
model_names = ['Llama-3.2-3B', 'Qwen2.5-3B', 'Llama-3.1-8B', 'Qwen2.5-7B']
batch_sizes = [2, 2, 1, 1]

zero_shot_types = ['fact', 'counter', 'other']

for model_name, batch_size in zip(model_names, batch_sizes):
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)

    for zero_shot_type in zero_shot_types:
        eval_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot_type}_sft_eval.json'
        eval_datas = json_utils.load_json(eval_file_path)

        save_dir = f'{out_dir}/{model_name}/zero_shot_{zero_shot_type}'
        evaluate_all(save_dir, eval_datas, batch_size)